# Step 04: download every round's topics

Records go into a **pool** shared by every experiment that names it (`[download] pool`), each stored once under its own ID. What an experiment picked stays in its own `runs/<experiment>/04-download/`, so experiments never download a paper twice but each uses only its own papers.

- **pubmed**: per topic, the open-access papers first (`max_fulltext`, with PMC full text as JATS XML), then the rest (`max_abstract_only`, abstracts only)
- **clinicaltrials**: ClinicalTrials.gov study records (they don't count against the paper budget; `max_studies` limits them)
- **arxiv**: arXiv papers and their HTML full text, for experiments outside biomedicine (`experiments/test_arxiv.toml`)

This notebook runs the whole chain from the command line on the examples experiment, two rounds, into `corpora/examples`. It starts the experiment's runs afresh but keeps the pool, so on a rerun the records are already there. It needs network access and an NCBI key (`.env.local` or `~/.ncbi.env`).

All the example notebooks use `experiments/examples.toml`: the default experiment with tiny limits and its own pool (`corpora/examples`). Run `00-setup` for it first: `python src/00-setup/run.py --experiment experiments/examples.toml --no-index`.

In [1]:
import gzip
import json
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

import duckdb
from lxml import etree

SRC = Path.cwd().parent
LACUNA = SRC.parent
EXPERIMENT = LACUNA / 'experiments' / 'examples.toml'
runs, pool = LACUNA / 'runs' / 'examples', LACUNA / 'corpora' / 'examples'
def writable(func, path, _):
    """Clear the read-only flag OneDrive puts on synced files, then delete again."""
    os.chmod(path, stat.S_IWRITE)
    func(path)


if runs.exists():
    shutil.rmtree(runs, onexc=writable)


def step(name, *extra):
    """Run one step's CLI on the examples experiment and show what it printed."""
    result = subprocess.run([sys.executable, SRC / name / 'run.py', '--experiment', EXPERIMENT, *extra],
                            capture_output=True, encoding='utf-8', errors='replace')
    print(result.stdout if result.returncode == 0 else result.stderr[-3000:])


for name in ['01-query', '02-expand', '03-topics']:
    step(name)

goal  new treatment avenues for glioblastoma
seed  'glioblastoma' -> mesh:D005909 (Glioblastoma)
wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\01-query\seeds.json (1 seeds)



wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\02-expand\graph_1 (17 nodes, 16 edges from 1 seeds)



glioblastoma                             pubmed   35,029 ->     3  clinicaltrials    2,279 ->     3
astrocytoma                              pubmed   11,278 ->     3  clinicaltrials    2,323 ->     3
glioma                                   pubmed   43,727 ->     3  clinicaltrials    3,587 ->     3
wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\03-topics\round_1.json (3 topics, 9 of 20,000 budget papers used across rounds)



## 1. Plan, then download round 1

`--plan` only counts; without it, step 04 downloads into the pool.

In [2]:
step('04-download', '--plan')
step('04-download')

examples: 1 rounds, 3 topics -> pool C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\corpora\examples
== pubmed
topic                                        round   matches open acc.
glioblastoma                                     1    35,029    10,252
astrocytoma                                      1    11,278       957
glioma                                           1    43,727     9,056
== clinicaltrials
would get glioblastoma: 3 studies
would get astrocytoma: 3 studies
would get glioma: 3 studies



examples: 1 rounds, 3 topics -> pool C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\corpora\examples
== pubmed
glioblastoma                                 round 1       3 papers
astrocytoma                                  round 1       3 papers
glioma                                       round 1       3 papers
9 distinct papers across 3 topics
pubmed: 9 on disk, 0 to fetch
7 of them have a PMC copy
pmc: 7 on disk, 0 to fetch
9 papers in C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\04-download\papers_pubmed.json (7 with full text)
manifest_pubmed.parquet: 24 papers, 0 new
== clinicaltrials
saved glioblastoma (3 studies, 0 new to the pool)
saved astrocytoma (3 studies, 0 new to the pool)
saved glioma (3 studies, 0 new to the pool)
manifest_clinicaltrials.parquet: 13 studies, 0 new
4 studies in C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\04-download\papers_clinicaltrials.json



## 2. What the pool holds

Raw records under `data/`, and a manifest per source at the root: the paper's public date (online, else issue, else PubMed listing), its MeSH headings and whether the full text is there; a trial's status, phases and MeSH-coded conditions. A PMC article keeps its sections.

In [3]:
print(duckdb.sql(f"SELECT pmid, pmcid, date, date_source, full_text, left(title, 50) AS title, len(mesh) AS mesh "
                 f"FROM '{(pool / 'manifest_pubmed.parquet').as_posix()}' ORDER BY date"))
print(duckdb.sql(f"SELECT nct_id, status, phases, first_posted, condition_mesh[1:2] AS condition_mesh, "
                 f"left(title, 45) AS title FROM '{(pool / 'manifest_clinicaltrials.parquet').as_posix()}'"))
for path in sorted((pool / 'data' / 'pmc').glob('*.xml.gz'))[:3]:
    article = etree.fromstring(gzip.open(path).read())
    print('PMC', path.name.split('.')[0], [t.text for t in article.iterfind('body/sec/title')][:5])

┌──────────┬─────────────┬────────────┬─────────────┬───────────┬────────────────────────────────────────────────────┬───────┐
│   pmid   │    pmcid    │    date    │ date_source │ full_text │                       title                        │ mesh  │
│ varchar  │   varchar   │  varchar   │   varchar   │  boolean  │                      varchar                       │ int64 │
├──────────┼─────────────┼────────────┼─────────────┼───────────┼────────────────────────────────────────────────────┼───────┤
│ 3276093  │ NULL        │ 1988-01-01 │ issue       │ false     │ Cysticercosis.                                     │    12 │
│ 24884915 │ PMC4036080  │ 2014-05-17 │ epub        │ true      │ A large de novo 9p21.3 deletion in a girl affected │    18 │
│ 28699883 │ PMC5628627  │ 2017-01-01 │ issue       │ true      │ Familial melanoma-astrocytoma syndrome: synchronou │    11 │
│ 29979130 │ PMC6261346  │ 2018-07-06 │ epub        │ true      │ Novel suprasellar location of desmoplastic in

## 3. Round 2 downloads only what is new

Steps 02 and 03 again make `graph_2` and `round_2.json` with the next topics; step 04 reuses round 1's selections and the pool's records and fetches only round 2's.

In [4]:
step('02-expand')
step('03-topics')
step('04-download')
papers = json.loads((runs / '04-download' / 'papers_pubmed.json').read_text())
print(f'the examples experiment uses {len(papers)} papers, {sum(1 for p in papers if p["pmcid"])} with full text')
print(papers[0])
trials = json.loads((runs / '04-download' / 'papers_clinicaltrials.json').read_text())
print(f'and {len(trials)} trials, which step 05 reads like papers:', trials[0])
print(json.dumps(json.loads((runs / '04-download' / 'pubmed_topics.json').read_text()), indent=1))

wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\02-expand\graph_2 (17 nodes, 16 edges from 1 seeds)



desmoplastic_cerebral_astrocytoma_of_infancy pubmed        5 ->     1
melanoma_astrocytoma_syndrome            pubmed        4 ->     3
diffuse_intrinsic_pontine_glioma         pubmed      247 ->     3  clinicaltrials      142 ->     3
wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\03-topics\round_2.json (3 topics, 16 of 20,000 budget papers used across rounds)



examples: 2 rounds, 6 topics -> pool C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\corpora\examples
== pubmed
glioblastoma                                 round 1       3 papers
astrocytoma                                  round 1       3 papers
glioma                                       round 1       3 papers
desmoplastic_cerebral_astrocytoma_of_infancy round 2       1 papers
melanoma_astrocytoma_syndrome                round 2       3 papers
diffuse_intrinsic_pontine_glioma             round 2       3 papers
16 distinct papers across 6 topics
pubmed: 15 on disk, 1 to fetch
  pubmed 1/1  ~60/min  failed batches 0
13 of them have a PMC copy
pmc: 13 on disk, 0 to fetch
16 papers in C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\04-download\papers_pubmed.json (13 with full text)
manifest_pubmed.parquet: 25 papers, 1 new
== clinicaltrials
have  glioblastoma
have  astrocytoma
have  glioma
saved diffuse_intrinsic_pontine_glioma (3 studies, 

## From the command line

```
python src/04-download/run.py --experiment experiments/default.toml --plan
python src/04-download/run.py --experiment experiments/default.toml
python src/04-download/run.py --experiment experiments/default.toml --prune   # after 03 --redo dropped topics
```
The pool: `corpora/<pool>/data/{pubmed,pmc,clinicaltrials}/` and `manifest_*.parquet`. The experiment's picks: `runs/<experiment>/04-download/` (`papers_<source>.json`, `pubmed_topics.json`, `selections/`). Every paper has an ID made of its source and that source's own ID (`pubmed:3276093`, `arxiv:2410.18710`), which is how step 05 knows which reader opens it.